# 2. Prepare windows and features — Member 2

Run notebook 01 first, then run this notebook top to bottom. This notebook builds and saves the feature table directly; there is no separate prepare command to run.

Two seconds = 100 readings. Windows do not overlap or cross subjects, invalid readings or timestamp discontinuities. Incomplete tails are counted and discarded. Activity labels do not define boundaries.

Majority voting assigns labels afterward; ties follow LABELS order. Purity is the winning fraction. Mixed windows remain in evaluation.

In [ ]:
from pathlib import Path
import os
import sys
# Find the project from the workspace, project folder, or notebooks folder.
ROOT = next((p for p in [Path.cwd(), Path.cwd() / 'harth_project', Path.cwd().parent]
             if (p / 'har_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the IMLProj workspace or harth_project folder.')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from har_utils import *

## Build the dataset

This cell recomputes the 2-second features and replaces the corresponding feature, window metadata and audit files. If features or windowing change, rerun notebook 03 so predictions stay consistent.

58 time features are used for milestone models. The extra 24 cached frequency features are reserved for notebook 05.

In [ ]:
subject_files()  # Read the existing subject CSVs from data/harth/.
record_environment()
X, meta = build_windows(seconds=2)
display(X.head()); display(meta.head())
print('Windows and cached features:', X.shape)
print('Milestone time features:', len(feature_columns(X)))
print('Mixed fraction:', (meta.purity < 1).mean())
display(pd.read_csv(ROOT / 'results/window_audit_2s.csv'))

## See exactly how the stream is split

These are actual windows saved by build_windows(), not separately invented boundaries. Alternating shaded blocks show 2-second windows. The top panel shows two sensor signals; the lower panel shows the original per-reading labels. Above each block are its majority label and purity.

Activity changes do not move the boundaries. Each block becomes one feature-table row. The example selects consecutive windows from one uninterrupted part of a subject recording; gaps elsewhere restart windowing.

In [ ]:
# Choose consecutive saved windows without using activity labels.
subject = meta.subject.iloc[0]
subject_windows = meta[meta.subject == subject].sort_values('start_row').copy()
row_break = subject_windows.start_row.diff().ne(100)
time_break = pd.to_datetime(subject_windows.start_time).diff().dt.total_seconds().sub(2).abs().gt(.001)
blocks = (row_break | time_break).cumsum()
longest_block = blocks.value_counts().idxmax()
shown = subject_windows[blocks == longest_block].head(4)
frame = read_subject(ROOT / 'data/harth' / f'{subject}.csv')
left = int(shown.start_row.iloc[0])
right = int(shown.start_row.iloc[-1]) + 100
segment = frame.iloc[left:right]
seconds = np.arange(len(segment)) / 50

# The lower panel shows the original activity at every reading.
active_labels = [label for label in LABELS if label in segment.label.unique()]
label_positions = {label: i for i, label in enumerate(active_labels)}
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
axes[0].plot(seconds, segment.back_x, label='back x', linewidth=1)
axes[0].plot(seconds, segment.thigh_x, label='thigh x', linewidth=1)
axes[0].set_ylabel('Acceleration (g)')
axes[0].legend(loc='lower right')
axes[1].step(seconds, segment.label.map(label_positions), where='post', color='black')
axes[1].set_yticks(range(len(active_labels)), [ACTIVITIES[label] for label in active_labels])
axes[1].set_ylim(-.5, len(active_labels) - .5)

for number, row in enumerate(shown.itertuples(), start=1):
    begin = (row.start_row - left) / 50
    end = begin + 2
    color = 'tab:blue' if number % 2 else 'tab:orange'
    for ax in axes:
        ax.axvspan(begin, end, color=color, alpha=.12)
        ax.axvline(begin, color='grey', linestyle='--', linewidth=.8)
    axes[0].text((begin + end) / 2, 1.02,
        f'W{number}: {ACTIVITIES[row.label]} ({row.purity:.0%})',
        transform=axes[0].get_xaxis_transform(), ha='center', fontsize=9)
axes[-1].axvline(end, color='grey', linestyle='--', linewidth=.8)
axes[-1].set_xlim(0, end)
axes[-1].set_xlabel('Seconds from the first displayed window')
fig.suptitle(f'{subject}: each shaded block = 100 readings = 2 seconds', y=1.02)
plt.tight_layout(); plt.show()

display(shown[['subject', 'start_row', 'label', 'purity']].assign(
    end_row_exclusive=shown.start_row + 100,
    activity=shown.label.map(ACTIVITIES)))


## A closer look at a mixed-activity window

For illustration only, select one mixed window if present. This selection does not change the training data or segmentation. The bar chart shows the votes behind its majority label. If there are no mixed windows, show a pure window instead.

In [ ]:
mixed = meta[meta.purity < 1]
example_window = mixed.iloc[0] if len(mixed) else meta.iloc[0]
example_frame = read_subject(ROOT / 'data/harth' / f'{example_window.subject}.csv')
start = int(example_window.start_row)
readings = example_frame.iloc[start:start+100]
votes = readings.label.value_counts()
winning_label = int(example_window.label)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t = np.arange(100) / 50
for label in LABELS:
    mask = readings.label.to_numpy() == label
    if mask.any():
        axes[0].scatter(t[mask], readings.back_x.to_numpy()[mask],
                        s=12, label=ACTIVITIES[label])
axes[0].set(xlabel='Seconds within this window', ylabel='Back x (g)',
            title='100 readings, colored by their original activity')
axes[0].legend(fontsize=8)
axes[1].bar([ACTIVITIES[label] for label in votes.index], votes.to_numpy(),
            color=['tab:green' if label == winning_label else 'tab:grey'
                   for label in votes.index])
axes[1].set(ylabel='Number of readings', title='Majority vote gives one window label')
axes[1].tick_params(axis='x', labelrotation=25)
plt.tight_layout(); plt.show()
print('Saved label:', ACTIVITIES[winning_label])
print('Purity:', example_window.purity, '| winning votes / 100:', votes[winning_label] / 100)

## Calculate one example

Magnitude = sqrt(x²+y²+z²); RMS = sqrt(mean(signal²)); SMA = mean(|x|+|y|+|z|). Seven statistics on six axes and two magnitudes, plus two SMA features, give 58 time features.

In [ ]:
first = meta.iloc[0]
frame = read_subject(ROOT / 'data/harth' / (first.subject + '.csv'))
start = int(first.start_row)
window = frame.iloc[start:start+100][AXES].to_numpy()
print('Back x mean:', window[:, 0].mean())
print('Back x RMS:', np.sqrt(np.mean(window[:, 0] ** 2)))
print('Back mean magnitude:', np.linalg.norm(window[:, :3], axis=1).mean())
print('Back SMA:', np.abs(window[:, :3]).sum(axis=1).mean())
print('Stored mean:', X.iloc[0]['back_x__mean'])

## From one window to a feature row

The plot below uses the same 100 readings as the manual calculation above. The horizontal mean, mean ± standard deviation band, minimum, maximum and RMS summarize that signal. The table is the corresponding part of one feature-table row. Features summarize readings; they do not create new window boundaries.

In [ ]:
signal = window[:, 0]  # Back x from the manually inspected window above.
time_in_window = np.arange(len(signal)) / 50
mean = signal.mean()
std = signal.std()
rms = np.sqrt(np.mean(signal ** 2))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time_in_window, signal, label='Back x readings')
ax.axhspan(mean - std, mean + std, alpha=.15, color='tab:orange', label='Mean ± std')
ax.axhline(mean, color='tab:orange', label='Mean')
ax.axhline(rms, color='tab:green', linestyle=':', label='RMS')
ax.axhline(signal.min(), color='grey', linestyle='--', label='Min / max')
ax.axhline(signal.max(), color='grey', linestyle='--')
ax.set(xlabel='Seconds within one 2-second window', ylabel='Acceleration (g)',
       title='100 readings become summary features')
ax.legend(ncol=3); plt.tight_layout(); plt.show()
columns = [f'back_x__{stat}' for stat in ['mean', 'std', 'min', 'max', 'median', 'range', 'rms']]
display(X.loc[[0], columns])

In [ ]:
meta.purity.hist(bins=20)
plt.xlabel('Label purity'); plt.ylabel('Windows'); plt.show()
print('2-second windows:', len(meta))
print('Mixed fraction:', (meta.purity < 1).mean())

## Handoff to Member 3

Share results/features_2s.pkl and results/windows_2s.csv, with their audits and recorded package versions, or use the same project folder. Notebook 03 loads these files; it does not depend on this notebook's kernel variables.

StandardScaler is fitted inside each training fold in notebook 03. Do not standardize the full feature table here. PCA remains later work.